# NewsPrism: summarisation and framing analysis without large language models

This notebook provides a non-LLM counterpart to `newsprism_summary_comparison-1.ipynb`. Every method used here runs locally, requires no API key and has no usage quota. Four outputs are produced:
* extractive summaries of each article (Lead-3, TF-IDF centroid and TextRank);
* abstractive summaries of each article from BART fine-tuned on CNN/DailyMail, a summarisation model that is trained rather than prompted;
* a dominant frame for each article, assigned by zero-shot natural language inference (NLI) using the same frame labels as the LLM notebook;
* a topic-level comparison of coverage by political stance (or by outlet), consisting of distinctive vocabulary, frame counts and an extractive summary in which every sentence is cited to its source article.

**Relationship to the LLM notebook.** Text preparation is identical to section 3 of the LLM notebook, so sentence identifiers (`s001`, `s002`, and so on) refer to the same sentences in both notebooks. The frame labels are also identical. Outputs are saved separately in `outputs/non_llm/`, which allows the two approaches to be compared directly in the report.

| Method | Type | What it provides |
| --- | --- | --- |
| Lead-3 | Extractive | The first three sentences; a strong baseline for news |
| TF-IDF centroid | Extractive | The three sentences closest to the article's average TF-IDF vector |
| TextRank | Extractive, graph-based | The three most central sentences in a sentence-similarity graph (Mihalcea and Tarau, 2004) |
| BART-large-CNN | Abstractive, fine-tuned | A fluent summary written in new words (Lewis et al., 2020) |
| Zero-shot NLI | Classification | A frame label without task-specific training (Yin et al., 2019) |
| Weighted log-odds | Lexical comparison | Words used distinctively by each group (Monroe, Colaresi and Quinn, 2008) |

**Runtime.** Sections 1 to 3 and 6 run in seconds. Sections 4 and 5 download two models of about 1.6 GB each on first use. On an Apple M1 laptop, BART then takes about 9 seconds per article and zero-shot classification about 5 seconds, so the full corpus takes roughly one hour. `LIMIT` can be set to a small number for a trial run. Results are cached, so an interrupted run resumes where it stopped.

## 1. Setup and data loading
The corpus is loaded from the shared processed dataset. Paths are resolved relative to the repository root, so the notebook runs regardless of the folder in which the kernel starts.

`STANCE_PRED_PATH` should point to the political leaning predictions once they are available. Until then, section 6 compares outlets instead of stances.

In [4]:
import re, json, hashlib
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from IPython.display import Markdown, display
nltk.download("punkt_tab", quiet=True)  # sentence-splitter data, downloaded once

# Find the repo root by walking up from wherever the notebook was started, so the paths work from any folder.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed").is_dir())
DATA_PATH = ROOT / "data" / "processed" / "FINAL_newsprism_corpus.csv"
OUT_DIR = ROOT / "notebooks" / "05_framing_summarisation" / "outputs" / "non_llm"; OUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_ART_PATH = OUT_DIR.parent / "article_analysis.json"  # LLM results from the main notebook, used for comparison if present
STANCE_PRED_PATH = None  # the political leaning predictions CSV once available, for example ROOT / "data" / "enriched" / "political_leaning.csv"
STANCE_COLUMN = "political_leaning"  # column in that file holding left / centre / right
LIMIT = None  # set to a small number such as 10 for a quick trial run of sections 4 and 5
pd.set_option("display.max_colwidth", 90)

def load_json(path): return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
def save_json(path, obj): path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

In [5]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["published_at"]).dt.date  # most timestamps are date-only, so treat dates as approximate
df["event_group"] = df["event_id"]  # event merges are not needed in this notebook
print("articles:", len(df), "| topics:", df["topic"].value_counts().to_dict())
display(df.groupby(["topic", "publisher"]).size().unstack(fill_value=0))

articles: 253 | topics: {'clean_energy': 96, 'housing': 92, 'immigration': 65}


publisher,7NEWS,ABC News,News.com.au,SBS News,Sydney Morning Herald,The Age,The Guardian Australia
topic,,,,,,,
clean_energy,2,15,2,6,16,15,40
housing,6,16,13,12,21,16,8
immigration,8,15,4,2,16,5,15


## 2. Text preparation
The cleaning and sentence segmentation are identical to section 3 of the LLM notebook: promotional lines are removed, ABC "In short" key points are set aside as `publisher_summary` for use as a reference in the evaluation, and each remaining sentence receives an identifier. Because the procedure is identical, a sentence identifier refers to the same sentence in both notebooks.

In [6]:
PROMO_INLINE = re.compile(r"WATCH (?:THE VIDEO )?ABOVE:.{0,150}?Know the news with the 7NEWS app: Download today|Know the news with the 7NEWS app: Download today|Stream free on\s*$|News that makes sense Your trusted source for staying up-to-date with the world around you\.|Get free daily news updates and analysis, straight to your inbox\.|Loading\.\.\.")
PROMO_SENT = re.compile(r"(?i)^(sign up|subscribers can sign up|for the latest from sbs news|listen to australian and world news)|newsletter (delivers|is a (weekly )?wrap|is our guide)|sbs news podcasts|subscribe to abc news daily")
TITLE_SUFFIX = re.compile(r"\s+[-|]\s+(SMH\.com\.au|The Age|abc\.net\.au|ABC News.*|sbs\.com\.au|News\.com\.au|7NEWS|The Guardian.*)$")

def split_sentences(text):
    try: return sent_tokenize(text)
    except LookupError: return re.split(r"(?<=[.!?])\s+(?=[A-Z“\"‘])|(?<=[.!?][”\"’])\s+(?=[A-Z“\"‘])", text)  # simple fallback if NLTK data is missing

def prepare(row):
    # Turn one dataset row into the article dict used by every later step (same as the LLM notebook).
    text = PROMO_INLINE.sub(" ", row["clean_text"]); publisher_summary = None
    if text.startswith("In short"):  # ABC key points: keep as a reference summary, remove from the input
        s = split_sentences(re.sub(r"What[’']s next\?", " ", re.sub(r"^In short:?", "", text)).strip())
        publisher_summary, text = " ".join(s[:3]), " ".join(s[3:])
    kept = [s.strip() for s in split_sentences(text) if s.strip() and not PROMO_SENT.search(s)]
    sents = {f"s{i:03d}": s for i, s in enumerate(kept, 1)}
    body = " ".join(kept)
    return {"article_id": row["article_id"], "event_id": row["event_id"], "topic": row["topic"], "publisher": row["publisher"], "title": TITLE_SUFFIX.sub("", row["title"]), "date": str(row["date"]), "url": row["url"], "sents": sents, "text": body, "publisher_summary": publisher_summary, "text_sha1": hashlib.sha1(body.encode("utf-8")).hexdigest()}

arts = {row["article_id"]: prepare(row) for _, row in df.iterrows()}
todo_ids = list(arts)[:LIMIT] if LIMIT else list(arts)  # the articles processed by the slower sections 4 and 5
print("articles prepared:", len(arts), "| with a publisher summary:", sum(a["publisher_summary"] is not None for a in arts.values()), "| processed in sections 4 and 5:", len(todo_ids))

articles prepared: 253 | with a publisher summary: 28 | processed in sections 4 and 5: 253


## 3. Extractive summaries
Three extractive methods select three sentences from each article and present them in their original order. Because every sentence is copied from the article, extractive summaries are faithful by construction and each sentence carries its own identifier as evidence.
- **Lead-3** selects the first three sentences, exploiting the inverted-pyramid structure of news writing.
- **TF-IDF centroid** selects the three sentences most similar to the article's mean TF-IDF vector.
- **TextRank** builds a graph in which sentences are nodes and edges are weighted by TF-IDF cosine similarity, then ranks sentences by PageRank (Mihalcea and Tarau, 2004). A sentence ranks highly when it is similar to many other sentences that themselves rank highly.

In [7]:
K = 3  # sentences per extractive summary

def lead_k(sents, k=K): return list(sents.items())[:k]

def tfidf_centroid(sents, k=K):
    ids, texts = list(sents), list(sents.values())
    if len(texts) <= k: return list(sents.items())
    X = TfidfVectorizer(stop_words="english").fit_transform(texts)
    centroid = np.asarray(X.mean(axis=0))  # the average sentence vector for this article
    scores = cosine_similarity(X, centroid).ravel()  # how close each sentence is to that average
    top = sorted(np.argsort(scores)[::-1][:k])  # best k, put back in reading order
    return [(ids[i], texts[i]) for i in top]

def textrank(sents, k=K, d=0.85, iters=50):
    # PageRank over a graph of sentences, with edges weighted by TF-IDF cosine similarity.
    items = list(sents.items()); n = len(items)
    if n <= k: return items
    try: X = TfidfVectorizer(stop_words="english").fit_transform([s for _, s in items])
    except ValueError: return items[:k]  # no usable words, for example a very short transcript
    S = cosine_similarity(X); np.fill_diagonal(S, 0)
    out = S.sum(axis=1, keepdims=True)
    P = np.divide(S, out, out=np.full_like(S, 1 / n), where=out > 0)  # row-normalised transition matrix; isolated sentences link to all
    r = np.full(n, 1 / n)
    for _ in range(iters): r = (1 - d) / n + d * P.T @ r
    top = sorted(np.argsort(-r)[:k])
    return [items[i] for i in top]

EXTRACTIVE = {"lead-3": lead_k, "tf-idf centroid": tfidf_centroid, "textrank": textrank}
extractive = {aid: {name: [{"sentence_id": sid, "text": s} for sid, s in fn(a["sents"])] for name, fn in EXTRACTIVE.items()} for aid, a in arts.items()}
save_json(OUT_DIR / "extractive_summaries.json", extractive)
print("extractive summaries saved for", len(extractive), "articles")

example = arts[todo_ids[0]]
display(Markdown(f"### {example['publisher']}: {example['title']}\n\n" + "\n\n".join(f"**{name}:** " + " ".join(f"[{x['sentence_id']}] {x['text']}" for x in extractive[example["article_id"]][name]) for name in EXTRACTIVE)))

extractive summaries saved for 253 articles


### 7NEWS: 7NEWS Spotlight: Pauline Hanson’s UK tour and meeting with Tommy Robinson exposes One Nation’s creative accounting on immigration numbers

**lead-3:** [s001] Pauline Hanson has rarely had a better opportunity to get her facts right than in 7NEWS Spotlight’s special report broadcast on Sunday night. [s002] Months in production, with unprecedented access to Hanson, it included her first major television interview since calling for an Australian ‘monoculture’ at the National Press Club in June. [s003] The interview was conducted in London, by correspondent Liam Bartlett, midway through Hanson’s so-called fact-finding tour of the UK during which the Spotlight crew also filmed her meeting a who’s-who of Britain’s far right.

**tf-idf centroid:** [s011] On a podcast with British far-right extremist Tommy Robinson, filmed ‘fly-on-the-wall’ by Spotlight, Hanson claimed Australia was taking in 739,000 migrants a year. [s015] But what makes the number even harder to pin down is that Hanson fails to mention the 219,000 people who left Australia in the same year. [s042] When Robinson discussed the number of Muslims in Australia with Hanson — “you’ve got about 4 per cent... that’s a million?

**textrank:** [s011] On a podcast with British far-right extremist Tommy Robinson, filmed ‘fly-on-the-wall’ by Spotlight, Hanson claimed Australia was taking in 739,000 migrants a year. [s015] But what makes the number even harder to pin down is that Hanson fails to mention the 219,000 people who left Australia in the same year. [s042] When Robinson discussed the number of Muslims in Australia with Hanson — “you’ve got about 4 per cent... that’s a million?

## 4. Abstractive summaries with BART
`facebook/bart-large-cnn` is a sequence-to-sequence transformer fine-tuned on the CNN/DailyMail news summarisation dataset (Lewis et al., 2020). Unlike the extractive methods, it writes new sentences. Unlike an LLM, it is not prompted: it was trained for this single task, so it cannot follow instructions about attribution, neutrality or output format.

Two properties must be considered when interpreting its output:
- **Input length.** BART reads at most 1,024 tokens. Longer articles are truncated, and the number of truncated articles is reported. Because news places the main facts first, truncation usually removes background rather than the main event.
- **Faithfulness.** Abstractive models can introduce statements that the source does not support. To provide evidence for each summary sentence, it is aligned to the most similar source sentence using ROUGE-L precision (the share of the summary sentence's words, in order, that appear in the source sentence). Summary sentences with low alignment are flagged for manual checking.

In [8]:
RUN_BART = True
BART_MODEL = "facebook/bart-large-cnn"  # "sshleifer/distilbart-cnn-12-6" is a faster, slightly weaker alternative
BART_PATH = OUT_DIR / "bart_summaries.json"
ALIGN_MIN = 0.5  # summary sentences below this ROUGE-L precision against every source sentence are flagged
aligner = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def align(summary, sents):
    # For each summary sentence, the most similar source sentence (its evidence) and how much of it is covered.
    rows = []
    for s in split_sentences(summary):
        best_sid, best = max(((sid, aligner.score(src, s)["rougeL"].precision) for sid, src in sents.items()), key=lambda x: x[1], default=(None, 0.0))
        rows.append({"sentence": s, "best_source": best_sid, "precision": round(best, 3), "status": "supported" if best >= ALIGN_MIN else "check"})
    return rows

if RUN_BART:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    bart_tok = AutoTokenizer.from_pretrained(BART_MODEL)
    bart = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL).to(DEVICE).eval()
    store = load_json(BART_PATH); todo = [aid for aid in todo_ids if store.get(aid, {}).get("text_sha1") != arts[aid]["text_sha1"]]
    print(f"device: {DEVICE} | to summarise: {len(todo)} | already saved: {len(todo_ids) - len(todo)}")
    for n, aid in enumerate(todo, 1):
        a = arts[aid]
        n_tokens = len(bart_tok(a["text"])["input_ids"])
        enc = bart_tok(a["text"], truncation=True, max_length=1024, return_tensors="pt").to(DEVICE)
        with torch.no_grad(): out = bart.generate(**enc)  # uses the model's own settings: beam search, 56 to 142 tokens
        summary = bart_tok.decode(out[0], skip_special_tokens=True).strip()
        store[aid] = {"summary": summary, "input_tokens": n_tokens, "truncated": n_tokens > 1024, "alignment": align(summary, a["sents"]), "model": BART_MODEL, "text_sha1": a["text_sha1"]}
        if n % 10 == 0 or n == len(todo): save_json(BART_PATH, store); print(f"  {n} of {len(todo)} done")
    bart_store = {aid: store[aid] for aid in todo_ids if aid in store}
    print("truncated articles:", sum(r["truncated"] for r in bart_store.values()), "of", len(bart_store))
    r = bart_store[todo_ids[0]]
    display(Markdown(f"### {arts[todo_ids[0]]['title']}\n\n**BART:** {r['summary']}\n\n" + "\n".join(f"- {x['sentence']} *(closest source: {x['best_source']}, precision {x['precision']}, {x['status']})*" for x in r["alignment"])))
else:
    bart_store = load_json(BART_PATH); print("BART skipped; saved summaries loaded:", len(bart_store))

/Users/linh/Documents/Applied NLP/AT2/nlp_at2_newsprism/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1216.27it/s]


device: mps | to summarise: 253 | already saved: 0
  10 of 253 done


KeyboardInterrupt: 

## 5. Frame classification with zero-shot NLI
Zero-shot classification recasts labelling as natural language inference (Yin et al., 2019). An NLI model, here `facebook/bart-large-mnli`, judges whether a premise (the article) entails a hypothesis (a sentence describing a frame, such as "This article is about costs, money, jobs or economic consequences"). The frame whose hypothesis receives the highest entailment probability is assigned. No frame-labelled training data is required.

The frame labels are those of the LLM notebook, based on Semetko and Valkenburg (2000) with the additions `security` and `policy_process`. The `other` label is omitted, as it cannot be expressed as a hypothesis.

The premise consists of the headline and the first `LEAD_SENTENCES` sentences, because NLI models accept a limited input length and news articles state their angle early. As evidence for the assigned frame, each of the first `EVIDENCE_SENTENCES` sentences is also scored against the winning hypothesis, and the highest-scoring sentence is recorded. Where LLM frame labels are available from the main notebook, agreement between the two methods is reported in section 7.

In [ ]:
RUN_ZERO_SHOT = True
NLI_MODEL = "facebook/bart-large-mnli"
FRAME_PATH = OUT_DIR / "zero_shot_frames.json"
LEAD_SENTENCES, EVIDENCE_SENTENCES = 5, 15
FRAME_HYPOTHESES = {  # one hypothesis per frame; wording matters, so changes should be tested on a sample
    "conflict": "This article is about conflict or disagreement between people, parties or groups.",
    "human_interest": "This article tells the personal story or experience of individuals.",
    "economic_consequences": "This article is about costs, money, jobs or economic consequences.",
    "responsibility": "This article is about who is responsible for causing or solving a problem.",
    "morality": "This article is about moral values, ethics or religion.",
    "security": "This article is about security, crime, policing or border protection.",
    "policy_process": "This article is about government policy, legislation or the political process.",
}

if RUN_ZERO_SHOT:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
    label_ids = {k.lower(): v for k, v in nli.config.label2id.items()}
    ENT, CON = label_ids["entailment"], label_ids["contradiction"]

    def entail_logits(premises, hypotheses):
        # Entailment logit for each premise and hypothesis pair.
        enc = nli_tok(premises, hypotheses, truncation="only_first", max_length=512, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad(): logits = nli(**enc).logits
        return logits[:, [CON, ENT]].float().cpu()  # contradiction and entailment only, as in the standard zero-shot method

    def classify_frame(a):
        frames, hyps = list(FRAME_HYPOTHESES), list(FRAME_HYPOTHESES.values())
        premise = a["title"] + ". " + " ".join(list(a["sents"].values())[:LEAD_SENTENCES])
        probs = entail_logits([premise] * len(hyps), hyps)[:, 1].softmax(0).numpy()  # one label per article: softmax over the entailment logits
        best = int(probs.argmax())
        lead = list(a["sents"].items())[:EVIDENCE_SENTENCES]
        ev = entail_logits([s for _, s in lead], [hyps[best]] * len(lead)).softmax(1)[:, 1].numpy() if lead else np.array([])
        return {"frame": frames[best], "confidence": round(float(probs[best]), 3), "scores": {f: round(float(p), 3) for f, p in zip(frames, probs)}, "evidence_sentence": lead[int(ev.argmax())][0] if len(ev) else None, "evidence_score": round(float(ev.max()), 3) if len(ev) else None, "model": NLI_MODEL, "text_sha1": a["text_sha1"]}

    store = load_json(FRAME_PATH); todo = [aid for aid in todo_ids if store.get(aid, {}).get("text_sha1") != arts[aid]["text_sha1"]]
    print(f"device: {DEVICE} | to classify: {len(todo)} | already saved: {len(todo_ids) - len(todo)}")
    for n, aid in enumerate(todo, 1):
        store[aid] = classify_frame(arts[aid])
        if n % 20 == 0 or n == len(todo): save_json(FRAME_PATH, store); print(f"  {n} of {len(todo)} done")
    frame_store = {aid: store[aid] for aid in todo_ids if aid in store}
else:
    frame_store = load_json(FRAME_PATH); print("Zero-shot skipped; saved frames loaded:", len(frame_store))

if frame_store:
    fr = pd.DataFrame([{"article_id": aid, "topic": arts[aid]["topic"], "publisher": arts[aid]["publisher"], **{k: v for k, v in r.items() if k in ("frame", "confidence", "evidence_sentence")}} for aid, r in frame_store.items()])
    display(pd.crosstab(fr["topic"], fr["frame"]))
    r = frame_store[todo_ids[0]]; a = arts[todo_ids[0]]
    print(f"\nExample: {a['title']}\n  frame: {r['frame']} ({r['confidence']}) | evidence [{r['evidence_sentence']}]: {a['sents'].get(r['evidence_sentence'], '')[:200]}")

## 6. Topic comparison by political stance
For each topic, articles are grouped by predicted political stance, and three complementary views are produced for each group. Where stance predictions are not yet available, the comparison is run by outlet instead, using the same code.
1. **Distinctive vocabulary.** The weighted log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi and Quinn, 2008) identifies the words and two-word phrases that one group uses more than the others. Each value is a z-score, so terms above 1.96 are distinctive at roughly the 5% level, and the prior prevents rare words from dominating. Each distinctive term is shown with a sentence from that group that uses it.
2. **Frame counts** from the zero-shot classification in section 5.
3. **An extractive group summary.** Sentences are pooled from all of the group's articles and scored for being representative of the group (similar to the group's TF-IDF centroid) and distinctive from the other groups (dissimilar to their centroids). Sentences are then selected with maximal marginal relevance, which penalises redundancy, and at most one sentence is taken from each article. Every summary sentence is quoted verbatim with its article and sentence identifier, so the evidence for each statement is the statement itself.

The caveats of the LLM notebook apply equally here. The articles concern different events, so differences may reflect story selection as much as framing. The stance labels are classifier predictions. Groups with fewer than `MIN_ARTICLES` articles are not described.

In [ ]:
GROUP_BY = "stance"  # "stance" once predictions are available; "publisher" compares outlets
MIN_ARTICLES = 5  # groups smaller than this are too thin to describe
STANCE_ALIASES = {"left": "left", "centre": "centre", "center": "centre", "right": "right"}  # extend once the classifier's label format is known

def load_stances():
    # article_id -> left / centre / right from the teammate's predictions file; empty until that file exists.
    if not STANCE_PRED_PATH or not Path(STANCE_PRED_PATH).exists(): return {}
    pred = pd.read_csv(STANCE_PRED_PATH)
    labels = {aid: STANCE_ALIASES.get(str(v).strip().lower()) for aid, v in zip(pred["article_id"], pred[STANCE_COLUMN]) if pd.notna(v)}
    unknown = Counter(str(v) for aid, v in zip(pred["article_id"], pred[STANCE_COLUMN]) if pd.notna(v) and labels.get(aid) is None)
    if unknown: print("labels not recognised (add them to STANCE_ALIASES):", dict(unknown))
    return {aid: s for aid, s in labels.items() if s}

stances = load_stances()
if GROUP_BY == "stance" and not stances:
    print("No stance predictions found (STANCE_PRED_PATH is not set or the file is missing), so this section compares outlets instead.")
    GROUP_BY = "publisher"
group_of = {aid: (stances.get(aid) if GROUP_BY == "stance" else a["publisher"]) for aid, a in arts.items()}
print("grouping by:", GROUP_BY, "| articles with a group:", sum(g is not None for g in group_of.values()))

In [ ]:
def log_odds_z(counts_g, counts_rest, prior):
    # Weighted log-odds ratio with an informative Dirichlet prior (Monroe et al., 2008), as z-scores.
    a0, n_g, n_r = prior.sum(), counts_g.sum(), counts_rest.sum()
    delta = np.log((counts_g + prior) / (n_g + a0 - counts_g - prior)) - np.log((counts_rest + prior) / (n_r + a0 - counts_rest - prior))
    return delta / np.sqrt(1 / (counts_g + prior) + 1 / (counts_rest + prior))

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
EXTRA_STOP = {"mr", "ms", "mrs", "dr", "said", "says", "say", "told", "according", "just", "like", "really", "lot", "lots", "think", "know", "going", "want", "got", "make", "way", "thing", "things", "good", "better", "new", "year", "years", "time", "people", "also", "including", "per", "cent"}  # reporting verbs, honorifics and filler

def distinctive_terms(topic_ids, groups, k=10, z_min=1.96):
    # Top terms per group compared with every other group in the topic, each with an example sentence from that group.
    cv = CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS | EXTRA_STOP), ngram_range=(1, 2), min_df=3, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]+\b")
    X = cv.fit_transform([arts[i]["text"] for i in topic_ids]); vocab = cv.get_feature_names_out()
    totals = np.asarray(X.sum(axis=0)).ravel().astype(float); out = {}
    for g, ids in groups.items():
        members = set(ids); mask = np.array([i in members for i in topic_ids])
        cg = np.asarray(X[mask].sum(axis=0)).ravel().astype(float)
        z = log_odds_z(cg, totals - cg, totals * 0.01 + 0.01)  # prior proportional to topic-wide counts
        terms = []
        for j in np.argsort(-z):
            if z[j] < z_min or len(terms) == k: break
            pattern = re.compile(r"\b" + re.escape(vocab[j]).replace(r"\ ", r"\W+") + r"\b", re.I)
            ex = next(((i, sid, s) for i in ids for sid, s in arts[i]["sents"].items() if pattern.search(s)), None)
            terms.append({"term": vocab[j], "z": round(float(z[j]), 2), "count": int(cg[j]), "example": {"article_id": ex[0], "sentence_id": ex[1], "text": ex[2]} if ex else None})
        out[g] = terms
    return out

def group_summary(groups, g, k=5, lam=0.5, mmr=0.7, min_words=8):
    # Extractive summary of one group: representative of it, distinct from the other groups, one sentence per article at most.
    pool = [(i, sid, s) for i in groups[g] for sid, s in arts[i]["sents"].items() if len(s.split()) >= min_words]
    others = [(i, sid, s) for h, ids in groups.items() if h != g for i in ids for sid, s in arts[i]["sents"].items() if len(s.split()) >= min_words]
    if not pool: return []
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit([s for *_, s in pool + others])
    P = vec.transform([s for *_, s in pool])
    own = cosine_similarity(P, np.asarray(P.mean(axis=0))).ravel()
    other_sims = [cosine_similarity(P, np.asarray(vec.transform([s for i, _, s in others if group_of[i] == h]).mean(axis=0))).ravel() for h in groups if h != g]
    other = np.max(other_sims, axis=0) if other_sims else np.zeros(len(pool))  # similarity to the closest other group
    score = own - lam * other
    chosen, used_articles = [], set()
    while len(chosen) < k:
        best, best_val = None, -np.inf
        for j in np.argsort(-score)[:300]:
            if pool[j][0] in used_articles: continue
            redundancy = max((cosine_similarity(P[j], P[c])[0, 0] for c in chosen), default=0.0)
            val = mmr * score[j] - (1 - mmr) * redundancy
            if val > best_val: best, best_val = j, val
        if best is None: break
        chosen.append(best); used_articles.add(pool[best][0])
    return [{"article_id": pool[j][0], "publisher": arts[pool[j][0]]["publisher"], "sentence_id": pool[j][1], "text": pool[j][2], "score": round(float(score[j]), 3)} for j in chosen]

GROUP_PATH = OUT_DIR / f"group_comparison_{GROUP_BY}.json"
results = {}
for topic in sorted(df["topic"].unique()):
    topic_ids = [aid for aid, a in arts.items() if a["topic"] == topic and group_of[aid] is not None]
    all_groups = {g: [i for i in topic_ids if group_of[i] == g] for g in sorted({group_of[i] for i in topic_ids})}
    groups = {g: ids for g, ids in all_groups.items() if len(ids) >= MIN_ARTICLES}
    thin = {g: len(ids) for g, ids in all_groups.items() if len(ids) < MIN_ARTICLES}
    if len(groups) < 2: print(f"{topic}: skipped, fewer than 2 groups have {MIN_ARTICLES} or more articles"); continue
    used_ids = [i for ids in groups.values() for i in ids]
    terms = distinctive_terms(used_ids, groups)
    results[topic] = {"group_by": GROUP_BY, "thin_groups": thin, "groups": {g: {"articles": len(ids), "outlets": dict(Counter(arts[i]["publisher"] for i in ids).most_common()), "frames": dict(Counter(frame_store[i]["frame"] for i in ids if i in frame_store).most_common()), "distinctive_terms": terms[g], "summary": group_summary(groups, g)} for g, ids in groups.items()}}
save_json(GROUP_PATH, results); print("saved:", GROUP_PATH.name)

def show_group_topic(topic, rec):
    parts = [f"### Topic: {topic} (grouped by {rec['group_by']})"]
    for g, r in rec["groups"].items():
        outlets = ", ".join(f"{p} {n}" for p, n in r["outlets"].items())
        frames = ", ".join(f"{f} {n}" for f, n in r["frames"].items()) or "not classified yet"
        terms = "; ".join(f"**{t['term']}** (z={t['z']})" for t in r["distinctive_terms"]) or "none above the threshold"
        summary = "\n".join(f"- “{s['text']}” ({s['publisher']}, {s['article_id']} {s['sentence_id']})" for s in r["summary"])
        parts.append(f"#### {g} ({r['articles']} articles: {outlets})\n\n*Frames:* {frames}\n\n*Distinctive terms:* {terms}\n\n*Summary:*\n{summary}")
    if rec["thin_groups"]: parts.append("*Too few articles to describe:* " + ", ".join(f"{g} ({n})" for g, n in rec["thin_groups"].items()))
    display(Markdown("\n\n".join(parts)))

for topic, rec in results.items(): show_group_topic(topic, rec)

## 7. Evaluation
Three automatic comparisons are reported:
1. **ROUGE against the publisher summary (ABC articles only).** All extractive methods, BART and, where available, the LLM summaries from the main notebook are compared with the ABC "In short" reference. As discussed in the LLM notebook, ROUGE rewards lexical overlap and penalises legitimate paraphrase, so it should be read alongside the human review.
2. **BART evidence alignment.** The share of BART summary sentences whose closest source sentence covers at least `ALIGN_MIN` of their words. Sentences below this threshold may be unsupported and should be checked manually.
3. **Frame agreement with the LLM.** Where the main notebook has assigned frames, the proportion of articles on which the zero-shot classifier and the LLM agree, together with a confusion table. Agreement between two automatic methods does not establish correctness, but systematic disagreement identifies frames whose definitions require attention.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
ref_arts = [a for a in arts.values() if a["publisher_summary"] and len(a["sents"]) >= 5]
llm = load_json(LLM_ART_PATH)
systems = {name: (lambda a, name=name: " ".join(x["text"] for x in extractive[a["article_id"]][name])) for name in EXTRACTIVE}
systems["bart"] = lambda a: bart_store.get(a["article_id"], {}).get("summary")
if llm: systems["llm"] = lambda a: llm.get(a["article_id"], {}).get("summary")
rows = [{"system": name, "article_id": a["article_id"], **{k: v.fmeasure for k, v in scorer.score(a["publisher_summary"], text).items()}} for name, fn in systems.items() for a in ref_arts if (text := fn(a))]
print("Articles with a publisher reference:", len(ref_arts), "(systems are scored only on the articles they have summarised)")
display(pd.DataFrame(rows).groupby("system").agg(articles=("article_id", "count"), rouge1=("rouge1", "mean"), rouge2=("rouge2", "mean"), rougeL=("rougeL", "mean")).round(3).sort_values("rouge1", ascending=False))

if bart_store:
    al = [x for r in bart_store.values() for x in r["alignment"]]
    print(f"\nBART alignment: {sum(x['status'] == 'supported' for x in al)} of {len(al)} summary sentences ({sum(x['status'] == 'supported' for x in al) / max(len(al), 1):.0%}) have a source sentence covering at least {ALIGN_MIN:.0%} of their words")

both = [aid for aid in frame_store if aid in llm]
if both:
    cmp = pd.DataFrame({"zero_shot": [frame_store[a]["frame"] for a in both], "llm": [llm[a]["dominant_frame"] for a in both]})
    print(f"\nFrame agreement with the LLM: {(cmp['zero_shot'] == cmp['llm']).mean():.0%} of {len(both)} articles")
    display(pd.crosstab(cmp["llm"], cmp["zero_shot"], rownames=["LLM frame"], colnames=["zero-shot frame"]))
else: print("\nNo articles have both a zero-shot frame and an LLM frame yet, so frame agreement is not reported.")

## Limitations
- **BART truncation.** BART reads at most 1,024 tokens, so the later parts of long articles are not summarised. The number of truncated articles is reported in section 4.
- **BART cannot follow instructions.** It may drop attribution, merge a quoted opinion into narration or adopt the source's wording. Evidence alignment flags sentences without a close source match, but a high alignment score indicates lexical overlap rather than correct meaning.
- **Domain shift.** BART was fine-tuned on American and British news from CNN and the Daily Mail, which differs in style from Australian outlets.
- **Zero-shot frames depend on hypothesis wording.** Small changes to the hypothesis sentences can change the assigned frame, and only the headline and lead sentences are classified. The `other` frame is not available. A manually labelled sample is required to estimate accuracy.
- **Extractive group summaries describe, but do not synthesise.** They show representative sentences from a group's coverage but cannot state a generalisation such as "this group tends to emphasise costs"; that interpretation rests with the reader, supported by the distinctive terms and frame counts.
- **Lexical statistics reflect story selection.** Distinctive terms often name the events a group covered rather than how it covered them. Terms should be interpreted together with the example sentences.
- **No country filter.** Unlike the LLM notebook, articles about events outside Australia are not removed, because no model here identifies the main country of an article.
- **Stance labels are classifier predictions**, so misclassified articles are described under the wrong stance, and a stance group dominated by one outlet largely reflects that outlet.